# GPR Field Data QC Viewer
**Lunar Leaper / pE PRO**

Load, visualize, and interactively process GPR data. Select a dataset and line using the dropdowns, then adjust processing parameters.

In [40]:
# -- Imports ------------------------------------------------------------------
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import gdp
from gdp.preprocessing.time_zero_correction import find_sample_above_threshold
from pathlib import Path

widgets.Widget.close_all()  # Clear any stale widget state

## 2. Select data
Use the dropdowns to choose a dataset folder and line. The data loads automatically on selection.

In [41]:
# Set the root directory for GPR data
GPR_ROOT = Path("../Data/GPR/Field data")

In [55]:
# -- Dataset and line selection --------------------------------------------------
def get_datasets():
    # Relative path of each DT1 parent folder from GPR_ROOT
    return sorted(set(
        str(p.parent.relative_to(GPR_ROOT))
        for p in GPR_ROOT.rglob("*.DT1")
    ))

def get_lines(dataset):
    return sorted(p.name for p in (GPR_ROOT / dataset).glob("*.DT1"))

datasets = get_datasets()

w_dataset = widgets.Dropdown(
    options=datasets,
    value="Skull cave" if "Skull cave" in datasets else datasets[0],
    description="Dataset:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="320px")
)
w_line = widgets.Dropdown(
    options=get_lines(w_dataset.value),
    description="Line:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px")
)

def on_dataset_change(change):
    w_line.options = get_lines(change["new"])

w_dataset.observe(on_dataset_change, names="value")
display(widgets.HBox([w_dataset, w_line]))


In [56]:
# -- Processing --------------------------------------------------
out_load = widgets.Output()

# Set by load_data(), read by the processing cell
original_data = None
info = sfreq = nyquist = time_axis = dist_axis = None
freq = suggested_dewow = None

def load_data(_=None):
    global original_data, info, sfreq, nyquist, time_axis, dist_axis, freq, suggested_dewow
    data_file = GPR_ROOT / w_dataset.value / w_line.value
    out_load.clear_output(wait=True)
    with out_load:
        print(f"Loading {data_file}...")
        gpr_data = gdp.load_dt1(str(data_file))
        original_data = np.asarray(gpr_data.data, dtype=np.float64)
        info = gpr_data.info

        n_samples     = info['samples']
        time_window   = info['Total_time_window']  # ns
        sfreq         = n_samples / time_window * 1000  # MHz
        nyquist       = sfreq / 2
        ns_per_sample = time_window / n_samples
        time_axis     = np.linspace(0, time_window, n_samples)
        dist_axis     = np.linspace(info['Start_pos'], info['Final_pos'], original_data.shape[1])

        freq  = info['Freq']
        s_pos = info['Start_pos']
        f_pos = info['Final_pos']
        units = info['Pos_units']
        step  = info['Step_size']
        print(f"Loaded: {original_data.shape[0]} samples x {original_data.shape[1]} traces")
        print(f"  Antenna freq  : {freq} MHz")
        print(f"  Sampling rate : {sfreq:.0f} MHz  (Nyquist: {nyquist:.0f} MHz)")
        print(f"  Time window   : {time_window:.0f} ns  ({n_samples} samples, {ns_per_sample:.3f} ns/sample)")
        print(f"  Distance      : {s_pos:.1f} - {f_pos:.1f} {units}  (step {step} {units})")

        # Suggested dewow window: ~2 pulse periods (2 / antenna_freq in ns -> samples)
        suggested_dewow = round(2 / freq * 1000 / ns_per_sample)
        print(f"\n  Suggested dewow window : {suggested_dewow} samples  (~2 pulse periods at {freq} MHz)")

        tz_header    = info['TZ_at_pt']
        tz_header_ns = tz_header * ns_per_sample
        median_trace = np.median(original_data, axis=1)
        tz_gdp       = find_sample_above_threshold(median_trace, threshold=0.15)
        tz_gdp_ns    = tz_gdp * ns_per_sample
        diff_s       = tz_gdp - tz_header
        diff_ns      = tz_gdp_ns - tz_header_ns
        print(f"\n-- Time zero comparison --")
        print(f"  Header (TZ_at_pt)  : sample {tz_header:.2f}  ->  {tz_header_ns:.1f} ns")
        print(f"  gdp (median trace) : sample {tz_gdp}  ->  {tz_gdp_ns:.1f} ns")
        print(f"  Difference         : {diff_s:.2f} samples  ({diff_ns:.1f} ns)")

        # Update slider ranges -- defined in the processing cell
        if 'update_sliders' in globals():
            update_sliders(nyquist, tz_header, suggested_dewow)

w_line.observe(load_data, names="value")
w_dataset.observe(load_data, names="value")
load_data()
display(out_load)

Output()

In [44]:
# -- Trace patching -----------------------------------------------------------
# Splices LINE04 and LINE05 into the currently loaded LINE03.
# Run this cell after loading LINE03. Reloading via the dropdown resets
# original_data to the raw file; re-run this cell to reapply the patches.

out_patch = widgets.Output()

def apply_patches():
    global original_data
    out_patch.clear_output(wait=True)
    with out_patch:
        if original_data is None:
            print("Load LINE03 first.")
            return

        patched = original_data.copy()
        n_samp  = patched.shape[0]

        # -- Patch 1: LINE04 -> LINE03 traces 55-70 (0-indexed) --------------
        # 27.5 m / 0.5 m spacing = trace 55; 35.0 m = trace 70; 16 traces
        p04 = GPR_ROOT / w_dataset.value / "LINE04.DT1"
        d04 = np.asarray(gdp.load_dt1(str(p04)).data, dtype=np.float64)
        n04 = d04.shape[1]
        print(f"LINE04: {n04} traces loaded from {p04.name}")
        if n04 != 16:
            print(f"  WARNING: expected 16 traces, got {n04}")
        rows = min(n_samp, d04.shape[0])
        patched[:rows, 55:55+n04] = d04[:rows, :]
        print(f"  -> patched into LINE03 traces 55-{55+n04-1} (0-indexed, {55*0.5:.1f}-{(55+n04-1)*0.5:.1f} m)")

        # -- Patch 2: LINE05 -> LINE03 traces 32-48 (0-indexed) --------------
        # 16.0 m / 0.5 m spacing = trace 32; 24.0 m = trace 48; 17 traces
        p05 = GPR_ROOT / w_dataset.value / "LINE05.DT1"
        d05 = np.asarray(gdp.load_dt1(str(p05)).data, dtype=np.float64)
        print(f"\nLINE05: {d05.shape[1]} traces loaded from {p05.name}")

        # Remove bad trace at index 9 (0-indexed)
        bad_idx = 9
        d05 = np.delete(d05, bad_idx, axis=1)
        print(f"  Removed trace index {bad_idx} -> {d05.shape[1]} traces remain")

        # Collected backwards: reverse trace order
        d05 = d05[:, ::-1]
        print(f"  Reversed trace order")

        n05 = d05.shape[1]
        if n05 != 17:
            print(f"  WARNING: expected 17 traces after edit, got {n05}")
        rows = min(n_samp, d05.shape[0])
        patched[:rows, 32:32+n05] = d05[:rows, :]
        print(f"  -> patched into LINE03 traces 32-{32+n05-1} (0-indexed, {32*0.5:.1f}-{(32+n05-1)*0.5:.1f} m)")

        original_data = patched
        print("\nDone. original_data updated with both patches.")
        print("Adjust any slider or re-run the processing cell to refresh the plot.")

btn_patch = widgets.Button(
    description="Apply patches",
    button_style="warning",
    layout=widgets.Layout(width="160px")
)
btn_patch.on_click(lambda _: apply_patches())

display(widgets.VBox([
    widgets.HTML("<b>Trace patching</b> &mdash; LINE04 and LINE05 into LINE03"),
    btn_patch,
    out_patch,
]))

In [45]:
# -- Line stitching -----------------------------------------------------------
# Concatenate multiple lines end-to-end into a single profile.
# Select lines in order (top = first), tick "flip" to reverse polarity of that
# segment (e.g. if antennas were oriented opposite), set overlap traces, stitch.
# Reloading via the dropdown resets original_data.

_STITCH_NONE = "---"

def _stitch_line_options():
    return [_STITCH_NONE] + get_lines(w_dataset.value)

_dd_style  = {"description_width": "60px"}
_dd_layout = widgets.Layout(width="240px")
_cb_layout = widgets.Layout(width="60px")

w_stitch_lines = [
    widgets.Dropdown(options=_stitch_line_options(), description=f"Line {i+1}:",
                     style=_dd_style, layout=_dd_layout)
    for i in range(6)
]
w_stitch_flips = [
    widgets.Checkbox(value=False, description="flip", indent=False, layout=_cb_layout)
    for _ in range(6)
]

# Default first two slots to first two available lines
_avail = get_lines(w_dataset.value)
if len(_avail) > 0: w_stitch_lines[0].value = _avail[0]
if len(_avail) > 1: w_stitch_lines[1].value = _avail[1]

w_stitch_overlap = widgets.IntSlider(
    value=1, min=0, max=20, step=1,
    description="Overlap traces:",
    continuous_update=False,
    style={"description_width": "120px"},
    layout=widgets.Layout(width="360px")
)

out_stitch = widgets.Output()

def apply_stitch(_=None):
    global original_data, dist_axis
    out_stitch.clear_output(wait=True)
    with out_stitch:
        selected = [
            (w.value, f.value)
            for w, f in zip(w_stitch_lines, w_stitch_flips)
            if w.value != _STITCH_NONE
        ]
        if len(selected) < 2:
            print("Select at least 2 lines to stitch.")
            return

        overlap   = int(w_stitch_overlap.value)
        segments  = []
        n_samp_min = None

        for i, (name, flip) in enumerate(selected):
            path = GPR_ROOT / w_dataset.value / name
            d = np.asarray(gdp.load_dt1(str(path)).data, dtype=np.float64)
            n_samp_min = d.shape[0] if n_samp_min is None else min(n_samp_min, d.shape[0])
            segments.append((name, d, 0 if i == 0 else overlap, flip))

        parts = []
        total_traces = 0
        for name, d, skip, flip in segments:
            chunk = d[:n_samp_min, skip:]
            if flip:
                chunk = -chunk
            parts.append(chunk)
            flip_str = " [polarity flipped]" if flip else ""
            print(f"  {name}: {d.shape[1]} traces, skipping first {skip} -> {chunk.shape[1]} used{flip_str}")
            total_traces += chunk.shape[1]

        stitched  = np.hstack(parts)
        step      = info['Step_size']
        new_dist  = np.arange(total_traces) * step + dist_axis[0]

        original_data = stitched
        dist_axis     = new_dist

        print(f"\nStitched {len(selected)} lines -> {total_traces} traces total")
        print(f"Distance: {new_dist[0]:.1f} - {new_dist[-1]:.1f} {info['Pos_units']}")
        if n_samp_min < max(d.shape[0] for _, d, _, _ in segments):
            print(f"WARNING: lines had different sample counts; truncated to {n_samp_min} samples.")
        print("\nDone. Adjust any slider to refresh the plot.")
        if 'update_sliders' in globals():
            update_sliders(nyquist, info['TZ_at_pt'], suggested_dewow)

btn_stitch = widgets.Button(
    description="Stitch lines",
    button_style="info",
    layout=widgets.Layout(width="140px")
)
btn_stitch.on_click(apply_stitch)

def _refresh_stitch_options(change):
    opts = _stitch_line_options()
    for w in w_stitch_lines:
        w.options = opts
w_dataset.observe(_refresh_stitch_options, names="value")

# Layout: each row = dropdown + flip checkbox
rows = [widgets.HBox([w_stitch_lines[i], w_stitch_flips[i]]) for i in range(6)]
display(widgets.VBox([
    widgets.HTML("<b>Line stitching</b>"),
    widgets.HBox([widgets.VBox(rows[:3]), widgets.VBox(rows[3:])]),
    w_stitch_overlap,
    btn_stitch,
    out_stitch,
]))

## 3. Interactive processing

Adjust the sliders below to process your GPR data in real-time. Use the preview window to assess data quality before finalizing parameters.

In [ ]:
# -- Processing function ------------------------------------------------------
def apply_processing(dewow_w, tzero_shift, flow, fhigh, gain_exp, stack_n):
    from gdp.preprocessing.filtering import dewow as dewow_fn, filter_data
    from gdp.preprocessing.gain import apply_gain as apply_gain_fn
    from scipy.ndimage import shift as ndshift

    processed = original_data.copy()

    # Stacking: average every stack_n consecutive traces before any filtering
    if stack_n > 1:
        n_samp, n_tr = processed.shape
        n_stacked = n_tr // stack_n
        processed = processed[:, :n_stacked * stack_n].reshape(n_samp, n_stacked, stack_n).mean(axis=2)

    processed = dewow_fn(processed, window_length=dewow_w)

    if tzero_shift != 0:
        processed = ndshift(processed, (tzero_shift, 0), order=1, mode='constant', cval=0)

    try:
        processed = filter_data(processed, (flow, fhigh), sfreq, 'bandpass', N=4)
    except Exception as e:
        print(f"Bandpass filter failed: {e}")

    try:
        processed, _ = apply_gain_fn(processed, sfreq, 'linear', exponent=gain_exp)
    except Exception as e:
        print(f"Gain failed: {e}")

    return processed

# -- Widget definitions -------------------------------------------------------
w_stack = widgets.IntSlider(
    value=1, min=1, max=50, step=1,
    description="Stack (traces):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px")
)
w_dewow = widgets.IntSlider(
    value=suggested_dewow, min=1, max=150, step=1,
    description="Dewow window (samples):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px")
)
w_tzero = widgets.FloatSlider(
    value=-info['TZ_at_pt'], min=-200, max=100, step=0.01,
    description="Time zero shift (samples):",
    continuous_update=False,
    style={"description_width": "180px"},
    layout=widgets.Layout(width="480px")
)
w_flow = widgets.FloatSlider(
    value=0.4*freq, min=1, max=freq, step=1,
    description="Low cutoff (MHz):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px")
)
w_fhigh = widgets.FloatSlider(
    value=2*freq, min=freq, max=nyquist*0.95, step=1,
    description="High cutoff (MHz):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px")
)
w_gain = widgets.FloatSlider(
    value=2.5, min=0., max=3.5, step=0.1,
    description="Gain exponent:",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px")
)
w_clip = widgets.FloatSlider(
    value=90, min=50, max=100, step=0.5,
    description="Clip percentile (%):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px"),
    readout_format=".1f"
)
w_xrange = widgets.FloatRangeSlider(
    value=[dist_axis[0], dist_axis[-1]],
    min=dist_axis[0], max=dist_axis[-1], step=0.1,
    description="X range (m):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px"),
    readout_format=".1f"
)
w_depth_toggle = widgets.ToggleButton(
    value=False,
    description="Show depth",
    button_style="",
    tooltip="Convert y-axis from two-way travel time (ns) to depth (m) using velocity below",
    layout=widgets.Layout(width="160px")
)
w_velocity = widgets.FloatSlider(
    value=0.13, min=0.05, max=0.3, step=0.005,
    description="Velocity (m/ns):",
    continuous_update=False,
    style={"description_width": "160px"},
    layout=widgets.Layout(width="480px"),
    readout_format=".3f"
)

out = widgets.Output()

# -- Slider update on dataset change ------------------------------------------
# ipywidgets validates min/max immediately. Reset min to 1 first so the new max
# can always be set safely regardless of which direction the antenna freq changes.
def update_sliders(nyquist, tz_header, suggested_dewow):
    _freq = info['Freq']
    w_flow.max   = _freq
    w_flow.value = 0.2 * _freq
    w_fhigh.min  = 1
    w_fhigh.max  = nyquist * 0.95
    w_fhigh.min  = _freq
    w_fhigh.value = 2 * _freq
    w_tzero.value = -tz_header
    w_dewow.value = suggested_dewow
    w_xrange.min  = dist_axis[0]
    w_xrange.max  = dist_axis[-1]
    w_xrange.value = [dist_axis[0], dist_axis[-1]]

# -- Plotting function --------------------------------------------------------
current_fig = None  # updated every render so the save cell can access it

def plot_comparison(dewow_w, tzero_shift, flow, fhigh, gain_exp, show_depth, velocity, stack_n, xrange, clip_pct):
    global current_fig
    from plotly.subplots import make_subplots

    processed = apply_processing(dewow_w, tzero_shift, flow, fhigh, gain_exp, stack_n)

    # Build dist_plot and orig_plot for stacked data
    if stack_n > 1:
        n_stacked = original_data.shape[1] // stack_n
        dist_plot = np.linspace(dist_axis[0], dist_axis[-1], n_stacked)
        n_samp, n_tr = original_data.shape
        orig_plot = original_data[:, :n_stacked * stack_n].reshape(n_samp, n_stacked, stack_n).mean(axis=2)
    else:
        dist_plot = dist_axis
        orig_plot = original_data

    # Crop to selected x range
    x0, x1 = xrange
    mask = (dist_plot >= x0) & (dist_plot <= x1)
    dist_plot = dist_plot[mask]
    orig_plot = orig_plot[:, mask]
    processed = processed[:, mask]

    vmax_orig = np.percentile(np.abs(orig_plot), clip_pct)
    vmax_proc = np.percentile(np.abs(processed), clip_pct)

    if show_depth:
        y_axis  = time_axis * velocity / 2
        y_label = "Depth (m)"
        y_hover = "Depth: %{y:.2f} m"
    else:
        y_axis  = time_axis
        y_label = "Two-way travel time (ns)"
        y_hover = "Time: %{y:.1f} ns"

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(f"Original (stacked x{stack_n})", "Processed"),
        horizontal_spacing=0.14,
    )

    common = dict(x=dist_plot, y=y_axis, colorscale='Gray', showscale=True)

    fig.add_trace(go.Heatmap(
        **common,
        z=orig_plot,
        zmin=-vmax_orig, zmax=vmax_orig,
        colorbar=dict(x=0.42, thickness=15, len=0.9, title="Ampl."),
        hovertemplate=f"Dist: %{{x:.1f}} m<br>{y_hover}<br>Amplitude: %{{z:.2e}}<extra>Original</extra>"
    ), row=1, col=1)

    fig.add_trace(go.Heatmap(
        **common,
        z=processed,
        zmin=-vmax_proc, zmax=vmax_proc,
        colorbar=dict(x=1.00, thickness=15, len=0.9, title="Ampl."),
        hovertemplate=f"Dist: %{{x:.1f}} m<br>{y_hover}<br>Amplitude: %{{z:.2e}}<extra>Processed</extra>"
    ), row=1, col=2)

    yaxis_style = dict(title=y_label, showgrid=False, autorange="reversed")
    vel_str = f"  |  v = {velocity:.3f} m/ns" if show_depth else ""
    stack_str = f"  |  Stack: {stack_n}" if stack_n > 1 else ""

    fig.update_layout(
        title=dict(
            text=(
                f"GPR Radargram{stack_str}  |  Dewow: {dewow_w} samples  |  "
                f"TZero: {tzero_shift}  |  Filter: {flow}-{fhigh} MHz  |  Gain exp: {gain_exp}{vel_str}"
            ),
            font=dict(size=12)
        ),
        height=600,
        plot_bgcolor="white",
        paper_bgcolor="white",
        hovermode="closest",
        margin=dict(r=80),
        yaxis=yaxis_style,
        yaxis2={**yaxis_style, "matches": "y"},
    )
    fig.update_xaxes(title_text=f"Distance ({info['Pos_units']})", showgrid=False)

    current_fig = fig  # store for save cell
    fig.show()

    rms_orig = np.sqrt(np.mean(orig_plot**2))
    rms_proc = np.sqrt(np.mean(processed**2))
    warn = "  (gain > 1: inflated)" if gain_exp > 1 else ""
    print(f"\nRMS = sqrt(mean(amplitude^2)) -- signal energy check")
    print(f"  Original : {rms_orig:.3e}")
    print(f"  Processed: {rms_proc:.3e}{warn}")

# -- Wire up widgets ----------------------------------------------------------
def on_change(_):
    out.clear_output(wait=True)
    with out:
        plot_comparison(
            int(w_dewow.value), float(w_tzero.value),
            float(w_flow.value), float(w_fhigh.value), float(w_gain.value),
            bool(w_depth_toggle.value), float(w_velocity.value),
            int(w_stack.value), w_xrange.value, float(w_clip.value)
        )

for w in [w_stack, w_dewow, w_tzero, w_flow, w_fhigh, w_gain, w_clip, w_xrange, w_depth_toggle, w_velocity]:
    w.observe(on_change, names="value")

# -- Display ------------------------------------------------------------------
display(widgets.VBox([
    widgets.HTML("<b>Stacking</b>"),
    w_stack,
    widgets.HTML("<b>Processing parameters</b>"),
    w_dewow, w_tzero, w_flow, w_fhigh, w_gain,
    widgets.HTML("<b>View</b>"),
    w_clip, w_xrange,
    widgets.HBox([w_depth_toggle, w_velocity]),
]))
display(out)
on_change(None)

Output()

In [47]:
# -- Save figure --------------------------------------------------------------
# Save PNG: redraws the current view with matplotlib (no extra packages needed).
# Save HTML: writes the interactive plotly figure as a standalone HTML file.

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

_SAVE_DIR = Path("../Results/GPR")

_default_name = str(_SAVE_DIR / f"{w_dataset.value.replace('/', '_')}_{w_line.value.replace('.DT1', '')}")

w_save_name = widgets.Text(
    value=_default_name,
    description="Filename:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="520px"),
    placeholder="no extension needed"
)

out_save = widgets.Output()

def _save_png():
    # Re-run processing with current slider values so matplotlib gets the same
    # data as the plotly view, without needing kaleido.
    stack_n   = int(w_stack.value)
    clip_pct  = float(w_clip.value)
    x0, x1   = w_xrange.value
    velocity  = float(w_velocity.value)
    show_depth = bool(w_depth_toggle.value)

    processed = apply_processing(
        int(w_dewow.value), float(w_tzero.value),
        float(w_flow.value), float(w_fhigh.value),
        float(w_gain.value), stack_n
    )

    # Build stacked original and matching dist axis
    if stack_n > 1:
        n_stacked = original_data.shape[1] // stack_n
        dist_plot = np.linspace(dist_axis[0], dist_axis[-1], n_stacked)
        n_samp, n_tr = original_data.shape
        orig_plot = original_data[:, :n_stacked * stack_n].reshape(n_samp, n_stacked, stack_n).mean(axis=2)
    else:
        dist_plot = dist_axis
        orig_plot = original_data

    # Crop to x range
    mask      = (dist_plot >= x0) & (dist_plot <= x1)
    dist_plot = dist_plot[mask]
    orig_plot = orig_plot[:, mask]
    processed = processed[:, mask]

    vmax_orig = np.percentile(np.abs(orig_plot), clip_pct)
    vmax_proc = np.percentile(np.abs(processed), clip_pct)

    y_axis  = time_axis * velocity / 2 if show_depth else time_axis
    y_label = "Depth (m)" if show_depth else "Two-way travel time (ns)"

    # extent: [left, right, bottom, top] — bottom > top so y increases downward
    ext_orig = [dist_plot[0], dist_plot[-1], y_axis[-1], y_axis[0]]
    ext_proc = [dist_plot[0], dist_plot[-1], y_axis[-1], y_axis[0]]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    ax1.imshow(orig_plot, aspect='auto', cmap='gray', extent=ext_orig,
               vmin=-vmax_orig, vmax=vmax_orig)
    ax1.set_title(f"Original (stacked x{stack_n})")
    ax1.set_xlabel(f"Distance ({info['Pos_units']})")
    ax1.set_ylabel(y_label)

    im = ax2.imshow(processed, aspect='auto', cmap='gray', extent=ext_proc,
                    vmin=-vmax_proc, vmax=vmax_proc)
    ax2.set_title("Processed")
    ax2.set_xlabel(f"Distance ({info['Pos_units']})")
    plt.colorbar(im, ax=ax2, label="Amplitude")

    stack_str = f"  |  Stack: {stack_n}" if stack_n > 1 else ""
    fig.suptitle(
        f"GPR Radargram{stack_str}  |  Dewow: {w_dewow.value}  |  "
        f"TZero: {w_tzero.value:.2f}  |  Filter: {w_flow.value:.0f}-{w_fhigh.value:.0f} MHz  |  "
        f"Gain: {w_gain.value}",
        fontsize=10
    )
    fig.tight_layout()

    stem = w_save_name.value.strip() or _default_name
    path = Path(stem).with_suffix(".png")
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(path), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved PNG: {path.resolve()}")

def _save_html():
    if current_fig is None:
        print("No figure to save yet — render a plot first.")
        return
    stem = w_save_name.value.strip() or _default_name
    path = Path(stem).with_suffix(".html")
    path.parent.mkdir(parents=True, exist_ok=True)
    current_fig.write_html(str(path))
    print(f"Saved HTML: {path.resolve()}")

def _on_save_png(_):
    out_save.clear_output(wait=True)
    with out_save:
        _save_png()

def _on_save_html(_):
    out_save.clear_output(wait=True)
    with out_save:
        _save_html()

btn_png  = widgets.Button(description="Save PNG",  button_style="success", layout=widgets.Layout(width="120px"))
btn_html = widgets.Button(description="Save HTML", button_style="",        layout=widgets.Layout(width="120px"))
btn_png.on_click(_on_save_png)
btn_html.on_click(_on_save_html)

display(widgets.VBox([
    widgets.HTML("<b>Save figure</b>"),
    w_save_name,
    widgets.HBox([btn_png, btn_html]),
    out_save,
]))